In [1]:
"""
====================================================================
STANDALONE NOTEBOOK -- STAIN-AUGMENTATION BASELINE, WITH CHECKPOINT-SAVING
Cross-Centre Generalization on Camelyon17-WILDS
====================================================================

Does ONE thing: trains + evaluates `baseline_stain_aug` (ResNet18, trained
END-TO-END with HED-space stain augmentation, directly for tumour
classification) on the 5 leave-one-centre-out (LOCO) folds, and -- unlike
every earlier run of this baseline -- SAVES each fold's trained weights to
disk. That's the fix: previously only the evaluation metrics survived
training, so this project's best-performing model could never be reloaded
for the probe battery / centre-signature probe / XAI study. Now it can.

No dependency on Notebook 2 or Notebook 3a/3b/3c/3d -- this only needs the
raw Camelyon17-WILDS dataset. Self-contained; paste and run.

COST: this is the expensive part of the whole project (5 full ResNet18
trainings from scratch, ~10 epochs each). If you can't finish all 5 folds
in one Kaggle session, set FOLDS_THIS_RUN below to a subset (e.g. [0, 1, 2]
for a first session), publish /kaggle/working/ as a dataset, then attach it
as input to a second run with FOLDS_THIS_RUN = [3, 4] -- this script
automatically finds and merges any already-completed folds from a prior
run (see `find_prior_run_dir` in main()) before doing the rest, so you
never lose completed folds and never redo them.

Needs `scikit-image` (HED stain augmentation) -- normally preinstalled on
Kaggle images; `pip install scikit-image` otherwise.
====================================================================
"""

import os
import pickle
import gc
import random
import warnings
from datetime import datetime

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torchvision.models as tvm
from sklearn.metrics import (
    accuracy_score, roc_auc_score, f1_score,
    precision_score, recall_score, balanced_accuracy_score,
    confusion_matrix, matthews_corrcoef,
)
from tqdm.auto import tqdm
from PIL import Image as PILImage
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')


# ====================================================================
# CONFIG
# ====================================================================
class Config:
    HF_DATASET = "wltjr1007/Camelyon17-WILDS"

    RESULTS_DIR = "/kaggle/working/stain_aug_results"
    CHECKPOINT_DIR = "/kaggle/working/stain_aug_checkpoints"

    ALL_CENTERS = [0, 1, 2, 3, 4]
    TEST_CENTER = 2   # tags this fold's row as 'primary' too, matching the rest of the project

    NUM_WORKERS = 2
    STAIN_AUG_EPOCHS = 10
    STAIN_AUG_LR = 1e-4
    STAIN_AUG_BATCH_SIZE = 64
    STAIN_AUG_INPUT_SIZE = 96
    STAIN_AUG_SEED = 42

    BOOTSTRAP_N_RESAMPLES = 1000
    BOOTSTRAP_SEED = 7


for _d in (Config.RESULTS_DIR, Config.CHECKPOINT_DIR):
    os.makedirs(_d, exist_ok=True)


def set_seed(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)


def worker_init_fn(worker_id):
    seed = torch.initial_seed() % (2 ** 32)
    np.random.seed(seed); random.seed(seed)


# ====================================================================
# DATA LOADING
# ====================================================================
def load_full_dataset():
    from datasets import load_dataset, concatenate_datasets
    ds = load_dataset(Config.HF_DATASET)
    parts = [ds[k] for k in ds.keys()]
    full = concatenate_datasets(parts) if len(parts) > 1 else parts[0]
    print(f"OK Loaded {len(full):,} patches")
    return full


class PatchDataset(Dataset):
    """Deterministic (no augmentation) transform -- used for EVAL only, so test-time
    results are reproducible."""
    def __init__(self, hf_dataset, indices, input_size):
        self.ds = hf_dataset
        self.indices = indices
        self.transform = transforms.Compose([
            transforms.Resize((input_size, input_size)),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ])

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        idx = self.indices[i]
        item = self.ds[int(idx)]
        img = item['image']
        if img.mode != 'RGB':
            img = img.convert('RGB')
        return {
            'image': self.transform(img),
            'label': int(item['label']),
            'center': int(item['center']),
            'patient': int(item['patient']),
        }


def hed_stain_augment(img_pil, alpha_range=(0.9, 1.1), beta_range=(-0.08, 0.08)):
    """
    Tellez et al. (2018) stain augmentation: perturb the image in HED (Haematoxylin-
    Eosin-DAB) colour space rather than plain RGB, since that's the axis along which
    real inter-hospital staining variation actually happens.
    """
    from skimage.color import rgb2hed, hed2rgb
    arr = np.asarray(img_pil).astype(np.float32) / 255.0
    hed = rgb2hed(arr)
    for c in range(3):
        alpha = np.random.uniform(*alpha_range)
        beta = np.random.uniform(*beta_range)
        hed[..., c] = hed[..., c] * alpha + beta
    rgb = np.clip(hed2rgb(hed), 0, 1)
    return PILImage.fromarray((rgb * 255).astype(np.uint8))


class StainAugDataset(Dataset):
    """Same interface as PatchDataset, but with heavy stain/colour augmentation when
    `train=True` and a plain deterministic resize+normalize when `train=False`."""
    def __init__(self, hf_dataset, indices, input_size, train=True):
        self.ds = hf_dataset
        self.indices = indices
        steps = []
        if train:
            steps += [transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(),
                      transforms.Lambda(lambda im: hed_stain_augment(im)),
                      transforms.ColorJitter(brightness=0.2, contrast=0.2,
                                            saturation=0.2, hue=0.02)]
        steps += [transforms.Resize((input_size, input_size)), transforms.ToTensor(),
                  transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])]
        self.transform = transforms.Compose(steps)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        idx = self.indices[i]
        item = self.ds[int(idx)]
        img = item['image']
        if img.mode != 'RGB':
            img = img.convert('RGB')
        return {'image': self.transform(img), 'label': int(item['label']),
                'center': int(item['center']), 'patient': int(item['patient'])}


def build_stain_aug_classifier():
    """ImageNet-initialized resnet18 with a fresh binary head -- trained end-to-end."""
    net = tvm.resnet18(weights=tvm.ResNet18_Weights.IMAGENET1K_V1)
    net.fc = nn.Linear(net.fc.in_features, 1)
    return net


# ====================================================================
# METRICS
# ====================================================================
def compute_metrics(y_true, y_pred, y_prob):
    m = {
        'accuracy': accuracy_score(y_true, y_pred),
        'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
        'auroc': roc_auc_score(y_true, y_prob),
        'f1': f1_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0),
        'mcc': matthews_corrcoef(y_true, y_pred),
    }
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    m['sensitivity'] = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    m['specificity'] = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    return m


def bootstrap_auroc_by_patient(y_true, y_prob, patient_ids,
                              n_boot=Config.BOOTSTRAP_N_RESAMPLES,
                              seed=Config.BOOTSTRAP_SEED):
    """Patient-level bootstrap 95% CI (patches from the same patient aren't
    independent samples, so this resamples PATIENTS with replacement, not patches)."""
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    patient_ids = np.asarray(patient_ids)
    unique_patients = np.unique(patient_ids)
    idx_by_patient = {p: np.where(patient_ids == p)[0] for p in unique_patients}

    rng = np.random.RandomState(seed)
    boot_aurocs = []
    for _ in range(n_boot):
        sampled = rng.choice(unique_patients, size=len(unique_patients), replace=True)
        idx = np.concatenate([idx_by_patient[p] for p in sampled])
        y_b, p_b = y_true[idx], y_prob[idx]
        if len(np.unique(y_b)) < 2:
            continue
        boot_aurocs.append(roc_auc_score(y_b, p_b))

    if not boot_aurocs:
        return {'auroc_boot_mean': float('nan'), 'auroc_ci_lo': float('nan'),
                'auroc_ci_hi': float('nan'), 'n_boot_valid': 0}
    lo, hi = np.percentile(boot_aurocs, [2.5, 97.5])
    return {'auroc_boot_mean': float(np.mean(boot_aurocs)), 'auroc_ci_lo': float(lo),
            'auroc_ci_hi': float(hi), 'n_boot_valid': len(boot_aurocs)}


# ====================================================================
# TRAIN / EVAL / LOCO (with the checkpoint-saving fix)
# ====================================================================
def train_stain_aug_model(hf_dataset, train_idx, device,
                         epochs=Config.STAIN_AUG_EPOCHS, lr=Config.STAIN_AUG_LR,
                         batch_size=Config.STAIN_AUG_BATCH_SIZE, seed=Config.STAIN_AUG_SEED,
                         tag='fold'):
    set_seed(seed)
    model = build_stain_aug_classifier().to(device)
    ds = StainAugDataset(hf_dataset, train_idx, Config.STAIN_AUG_INPUT_SIZE, train=True)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=True,
                        num_workers=Config.NUM_WORKERS, pin_memory=True,
                        worker_init_fn=worker_init_fn)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(1, epochs * len(loader)))
    criterion = nn.BCEWithLogitsLoss()

    model.train()
    for epoch in range(epochs):
        total_loss = 0.0
        for batch in tqdm(loader, desc=f"stain-aug train {tag} ep{epoch + 1}/{epochs}"):
            x = batch['image'].to(device, non_blocking=True)
            y = batch['label'].float().to(device, non_blocking=True)
            opt.zero_grad()
            logit = model(x).squeeze(1)
            loss = criterion(logit, y)
            loss.backward()
            opt.step()
            sched.step()
            total_loss += loss.item() * x.size(0)
        print(f"  [stain-aug {tag}] epoch {epoch + 1}/{epochs}  loss={total_loss / len(ds):.4f}")

    model.eval()
    return model


@torch.no_grad()
def evaluate_stain_aug_model(model, hf_dataset, test_idx, patients_all, device,
                            batch_size=64):
    ds = PatchDataset(hf_dataset, test_idx, Config.STAIN_AUG_INPUT_SIZE)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False,
                        num_workers=Config.NUM_WORKERS, pin_memory=True,
                        worker_init_fn=worker_init_fn)
    probs, labels = [], []
    for batch in tqdm(loader, desc="stain-aug eval"):
        x = batch['image'].to(device, non_blocking=True)
        logit = model(x).squeeze(1)
        probs.append(torch.sigmoid(logit).cpu().numpy())
        labels.append(batch['label'].numpy())
    probs = np.concatenate(probs)
    y_te = np.concatenate(labels)
    pat_te = patients_all[test_idx]
    preds = (probs >= 0.5).astype(int)

    m = compute_metrics(y_te, preds, probs)
    m.update(bootstrap_auroc_by_patient(y_te, probs, pat_te))
    return m


def run_stain_augmentation_loco(hf_dataset, centers_all, patients_all, device, folds):
    """
    Trains + evaluates baseline_stain_aug on `folds` (a subset of Config.ALL_CENTERS).
    THE FIX: every fold's model.state_dict() is saved to
    Config.CHECKPOINT_DIR/stain_aug_holdout{held_out}.pt, along with the
    train/test split it was trained on -- so it can be reloaded later exactly the
    way the reconstruction backbones' checkpoints are (see load_stain_aug_checkpoint).
    """
    print("\n" + "=" * 70)
    print(f"STAIN-AUGMENTATION BASELINE -- folds {folds} (checkpoint-saving enabled)")
    print("=" * 70)
    all_idx = np.arange(len(centers_all))
    rows = []
    for held_out in folds:
        train_centers = [c for c in Config.ALL_CENTERS if c != held_out]
        train_idx = all_idx[np.isin(centers_all, train_centers)]
        test_idx = all_idx[centers_all == held_out]
        print(f"\n{'-'*70}\n[stain-aug] HELD OUT centre {held_out}  |  TRAIN {train_centers}\n{'-'*70}")

        model = train_stain_aug_model(hf_dataset, train_idx, device, tag=f"holdout{held_out}")
        m = evaluate_stain_aug_model(model, hf_dataset, test_idx, patients_all, device)
        m.update({'model': 'baseline_stain_aug', 'train_centers': str(train_centers),
                 'test_center': int(held_out), 'split_tag': f'holdout_{held_out}',
                 'n_train': int(len(train_idx)), 'n_test': int(len(test_idx)),
                 'seed': Config.STAIN_AUG_SEED})
        rows.append(m)
        print(f"  [stain-aug] held-out {held_out}  AUROC={m['auroc']:.4f}  "
              f"[{m['auroc_ci_lo']:.4f}, {m['auroc_ci_hi']:.4f}]  BalAcc={m['balanced_accuracy']:.4f}")

        ckpt_path = f"{Config.CHECKPOINT_DIR}/stain_aug_holdout{held_out}.pt"
        torch.save({
            'state_dict': model.state_dict(),
            'held_out_centre': int(held_out),
            'train_centers': train_centers,
            'epochs': Config.STAIN_AUG_EPOCHS,
            'input_size': Config.STAIN_AUG_INPUT_SIZE,
            'seed': Config.STAIN_AUG_SEED,
            'auroc': m['auroc'],
        }, ckpt_path)
        print(f"    OK saved checkpoint -> {ckpt_path}")

        if held_out == Config.TEST_CENTER:
            primary_row = dict(m)
            primary_row['split_tag'] = 'primary'
            rows.append(primary_row)

        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    return rows


def load_stain_aug_checkpoint(held_out_centre, device):
    """Companion loader: rebuilds the same architecture and loads its trained
    weights, so this model can now be fed into the probe battery / XAI pipeline the
    same way the frozen reconstruction backbones are."""
    ckpt_path = f"{Config.CHECKPOINT_DIR}/stain_aug_holdout{held_out_centre}.pt"
    ckpt = torch.load(ckpt_path, map_location=device)
    model = build_stain_aug_classifier().to(device)
    model.load_state_dict(ckpt['state_dict'])
    model.eval()
    return model, ckpt


def make_summary_figure(df, prefix):
    prim = df[df['split_tag'] == 'primary']
    if not prim.empty:
        print(f"\nPrimary split (centre {Config.TEST_CENTER}): "
              f"AUROC={prim.iloc[0]['auroc']:.4f} "
              f"[{prim.iloc[0]['auroc_ci_lo']:.4f}, {prim.iloc[0]['auroc_ci_hi']:.4f}]")

    lco = df[df['split_tag'].str.startswith('holdout_', na=False)]
    if len(lco) == len(Config.ALL_CENTERS):
        fig, ax = plt.subplots(figsize=(7, 4))
        ax.bar(lco['test_center'].astype(str), lco['auroc'], color='#f39c12', edgecolor='black')
        yerr = np.abs(np.vstack([lco['auroc'] - lco['auroc_ci_lo'],
                                 lco['auroc_ci_hi'] - lco['auroc']]))
        ax.errorbar(lco['test_center'].astype(str), lco['auroc'], yerr=yerr,
                   fmt='none', ecolor='black', capsize=4)
        ax.set_xlabel('held-out centre')
        ax.set_ylabel('AUROC')
        ax.set_ylim(0.5, 1.0)
        ax.set_title('baseline_stain_aug -- LOCO AUROC by held-out centre')
        ax.grid(axis='y', alpha=0.3)
        plt.tight_layout()
        out_path = f"{prefix}_loco_bar.png"
        plt.savefig(out_path, dpi=150)
        plt.close(fig)
        print(f"OK {out_path}")
        print(f"\nLOCO mean AUROC = {lco['auroc'].mean():.4f}  "
              f"std = {lco['auroc'].std():.4f}")


# ====================================================================
# MAIN
# ====================================================================
# Edit this to a subset (e.g. [0, 1, 2]) if you can't finish all 5 folds in one
# Kaggle session -- the script automatically merges with any already-completed
# folds it finds from a prior run (see find_prior_run_dir below), so a second
# run with FOLDS_THIS_RUN = [3, 4] picks up where the first left off.
FOLDS_THIS_RUN = Config.ALL_CENTERS   # e.g. [0, 1, 2] for a first partial run


def main():
    set_seed(Config.STAIN_AUG_SEED)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Device: {device}")
    print("=" * 70)
    print("STANDALONE STAIN-AUGMENTATION BASELINE (with checkpoint-saving fix)")
    print(f"Folds this run: {FOLDS_THIS_RUN}")
    print("=" * 70)

    # ---- resume support: if a prior partial run's results were attached as an
    # input dataset (or already sit in /kaggle/working from an earlier cell in
    # this same session), load its completed rows + checkpoints so we don't
    # redo folds that already finished. ----
    def find_prior_run_dir():
        candidates = [
            "/kaggle/input/stain-aug-checkpoints-partial",  # <-- adjust slug if you published one
            "/kaggle/working",
        ]
        for c in candidates:
            rows_pkl = os.path.join(c, "stain_aug_rows_checkpoint.pkl")
            if os.path.exists(rows_pkl):
                return c
        return None

    completed_rows = []
    prior_dir = find_prior_run_dir()
    if prior_dir:
        with open(os.path.join(prior_dir, "stain_aug_rows_checkpoint.pkl"), 'rb') as f:
            completed_rows = pickle.load(f)
        done_folds = sorted({r['test_center'] for r in completed_rows
                            if r['split_tag'].startswith('holdout_')})
        print(f"OK found a prior partial run at {prior_dir} -- folds already done: {done_folds}")
        # also copy over its saved checkpoints if they're not already in this
        # session's working checkpoint dir (e.g. when resuming from an attached
        # input dataset rather than the same live session)
        prior_ckpt_dir = os.path.join(prior_dir, "stain_aug_checkpoints") if prior_dir != "/kaggle/working" \
            else Config.CHECKPOINT_DIR
        if os.path.isdir(prior_ckpt_dir) and prior_ckpt_dir != Config.CHECKPOINT_DIR:
            import shutil
            for fn in os.listdir(prior_ckpt_dir):
                if fn.endswith('.pt'):
                    shutil.copy2(os.path.join(prior_ckpt_dir, fn),
                               os.path.join(Config.CHECKPOINT_DIR, fn))
            print(f"OK copied prior checkpoints from {prior_ckpt_dir} -> {Config.CHECKPOINT_DIR}")
    else:
        done_folds = []
        print("No prior partial run found -- starting fresh.")

    folds_to_run = [f for f in FOLDS_THIS_RUN if f not in done_folds]
    if not folds_to_run:
        print("All requested folds are already done -- nothing to train this run.")
    else:
        print(f"Will train folds: {folds_to_run}")

    print("\nLoading Camelyon17-WILDS (raw images -- this trains end-to-end)...")
    hf_dataset = load_full_dataset()
    centers_all = np.array(hf_dataset['center'])
    patients_all = np.array(hf_dataset['patient'])

    new_rows = []
    if folds_to_run:
        new_rows = run_stain_augmentation_loco(
            hf_dataset, centers_all, patients_all, device, folds=folds_to_run)

    all_rows = completed_rows + new_rows
    # ---- always save the running checkpoint of rows, so a THIRD run (if needed)
    # can resume from here too ----
    with open(f"{Config.RESULTS_DIR}/stain_aug_rows_checkpoint.pkl", 'wb') as f:
        pickle.dump(all_rows, f)

    got_folds = sorted({r['test_center'] for r in all_rows if r['split_tag'].startswith('holdout_')})
    print(f"\nFolds completed so far (across all runs): {got_folds}")
    if got_folds != Config.ALL_CENTERS:
        missing = sorted(set(Config.ALL_CENTERS) - set(got_folds))
        print(f"FLAG not all 5 folds done yet -- missing {missing}. Publish this notebook's "
              f"/kaggle/working/ as a dataset, attach it as input to a new run, set "
              f"FOLDS_THIS_RUN = {missing}, and this script will pick up from here.")
    else:
        print("OK all 5 LOCO folds complete.")

    df = pd.DataFrame(all_rows)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    csv_path = f"{Config.RESULTS_DIR}/stain_aug_results_{timestamp}.csv"
    df.to_csv(csv_path, index=False)
    print(f"OK {csv_path}")

    make_summary_figure(df, f"{Config.RESULTS_DIR}/fig_{timestamp}")

    print("\n" + "=" * 70)
    print("DONE.")
    print(f"   Results CSV  : {csv_path}")
    print(f"   Checkpoints  : {Config.CHECKPOINT_DIR}  (one .pt per completed fold)")
    print("   Reload any fold's trained model with: load_stain_aug_checkpoint(held_out, device)")
    print("=" * 70)
    return df


df = main()

Device: cuda
STANDALONE STAIN-AUGMENTATION BASELINE (with checkpoint-saving fix)
Folds this run: [0, 1, 2, 3, 4]
No prior partial run found -- starting fresh.
Will train folds: [0, 1, 2, 3, 4]

Loading Camelyon17-WILDS (raw images -- this trains end-to-end)...


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00014.parquet:   0%|          | 0.00/470M [00:00<?, ?B/s]

data/train-00001-of-00014.parquet:   0%|          | 0.00/469M [00:00<?, ?B/s]

data/train-00002-of-00014.parquet:   0%|          | 0.00/489M [00:00<?, ?B/s]

...

  [stain-aug holdout4] epoch 9/10  loss=0.0454


stain-aug train holdout4 ep10/10:   0%|          | 0/4832 [00:00<?, ?it/s]

  [stain-aug holdout4] epoch 10/10  loss=0.0440


stain-aug eval:   0%|          | 0/2293 [00:00<?, ?it/s]

  [stain-aug] held-out 4  AUROC=0.9908  [0.9845, 0.9955]  BalAcc=0.9549
    OK saved checkpoint -> /kaggle/working/stain_aug_checkpoints/stain_aug_holdout4.pt

Folds completed so far (across all runs): [0, 1, 2, 3, 4]
OK all 5 LOCO folds complete.
OK /kaggle/working/stain_aug_results/stain_aug_results_20260927_113949.csv

Primary split (centre 2): AUROC=0.9826 [0.9531, 0.9920]
OK /kaggle/working/stain_aug_results/fig_20260927_113949_loco_bar.png

LOCO mean AUROC = 0.9883  std = 0.0041

DONE.
   Results CSV  : /kaggle/working/stain_aug_results/stain_aug_results_20260927_113949.csv
   Checkpoints  : /kaggle/working/stain_aug_checkpoints  (one .pt per completed fold)
   Reload any fold's trained model with: load_stain_aug_checkpoint(held_out, device)
